# ARIA LoRA fine-tune (free Colab, T4)

Fine-tunes Qwen2.5-7B-Instruct with QLoRA on Aiscern's ARIA dataset
(`aria_sft.jsonl`, built by `build_dataset.py`). Runs on Colab's free T4
(16GB) via 4-bit quantization.

**Before running:** upload `aria_sft.jsonl` to this Colab session (or
generate it in a cell below by uploading `aria-knowledge.json` and running
`build_dataset.py`).

**Runtime:** Runtime -> Change runtime type -> T4 GPU.


In [ ]:
!pip install -q -U transformers==4.46.3 peft==0.13.2 trl==0.12.1 \
    bitsandbytes==0.44.1 accelerate==1.1.1 datasets==3.1.0


## 1. Upload the dataset

Upload `aria_sft.jsonl` (built locally with `python build_dataset.py`), or the raw `aria-knowledge.json` to build it here.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick aria_sft.jsonl
DATASET_PATH = list(uploaded.keys())[0]
print(DATASET_PATH)


## 2. Load base model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False


## 3. Prepare the dataset

Renders each example through Qwen's chat template (tool-calling included), so the model trains directly on the exact format it will need to produce at inference.

In [ ]:
import json
from datasets import Dataset

def load_examples(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

raw = load_examples(DATASET_PATH)
print(f"{len(raw)} raw examples")

def render(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tools=example.get("tools"),
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(render, remove_columns=["messages", "tools"])
print(dataset[0]["text"][:500])


## 4. LoRA config + training

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./aria-lora-out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()


## 5. Save the adapter

Saves locally, and pushes to the Hugging Face Hub so the ZeroGPU Space (see `hf_space/app.py`) can pull it at load time — set `HF_REPO` and run `huggingface-cli login` (or paste a token below) first.

In [ ]:
ADAPTER_DIR = "./aria-lora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)


In [ ]:
# Optional: push to your own HF Hub repo so the Space can load it.
# from huggingface_hub import login
# login()  # paste an HF write token when prompted

HF_REPO = "your-username/aria-lora-qwen2.5-7b"  # <- change this

# trainer.model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)


## Next: deploy for inference

Copy `hf_space/app.py` and `hf_space/requirements.txt` into a new HF Space
(SDK: Gradio, hardware: ZeroGPU), set `BASE_MODEL` and `ADAPTER_REPO` env
vars in the Space settings to match what you used above, and it exposes an
OpenAI-compatible `/v1/chat/completions` endpoint — same tool-calling shape
this notebook trained on.